# Engine Comparison: Tournament vs Pool on Sphere

This notebook compares `GeneticEngine` with `TournamentSelection` vs `ElitePoolSelection` on the Sphere problem.

- **Benchmark**: Sphere function (continuous optimization)
- **Configurations**: Grid of `pop_size` × `mutation_rate` × `selection_param`
- **Runs per config**: 30 independent runs
- **Output**: Boxplots comparing selection strategies

## 1. Imports

In [ ]:
import os
from pathlib import Path
from typing import Any, Dict, Optional
from dataclasses import dataclass
import time

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.stats import mannwhitneyu

# MalthusJAX imports
from malthusjax.benchmarking.runner import BenchmarkRunner
from malthusjax.benchmarking.results import ExperimentResult, RunResult
from malthusjax.benchmarking.io import write_experiment_artifacts, read_summary_json
from malthusjax.engine.genetic_fastengine import GeneticEngine
from malthusjax.operators.selection.tournament import TournamentSelection
from malthusjax.operators.selection.elite_pool import ElitePoolSelection
from malthusjax.operators.crossover.real import BlendCrossover
from malthusjax.operators.mutation.real import GaussianMutation
from malthusjax.core.genome.real_genome import RealGenomeConfig
from malthusjax.core.fitness.real_evaluators import SphereConfig, SphereEvaluator

print("✓ All imports successful")

## 2. Configuration

In [ ]:
# Experiment parameters
NUM_GENERATIONS = 100
N_RUNS = 30  # Runs per condition for boxplots
DIM = 10
GLOBAL_SEED = 42

# Parameter grid
POP_SIZES = [50, 100]
MUTATION_RATES = [0.01, 0.05]
TOURNAMENT_SIZES = [2, 4, 8]
POOL_SIZES = [10, 20, 50]

# Output
RESULTS_DIR = Path("results/engine_comparison")
RESULTS_CSV = RESULTS_DIR / "boxplot_data.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Config: {NUM_GENERATIONS} gens, {N_RUNS} runs per config, dim={DIM}")
print(f"✓ Output dir: {RESULTS_DIR}")

## 3. Engine Factory

In [ ]:
def make_genetic_engine(
    selection_type: str,
    selection_param: int,
    pop_size: int,
    mutation_rate: float,
    num_generations: int,
    dim: int = DIM,
) -> GeneticEngine:
    """Create a GeneticEngine with specified configuration.
    
    Args:
        selection_type: 'tournament' or 'pool'
        selection_param: tournament_size or pool_size
        pop_size: population size
        mutation_rate: Gaussian mutation rate
        num_generations: total generations to run
        dim: problem dimension
    """
    # Genome config (Sphere: continuous [-5, 5] domain)
    genome_config = RealGenomeConfig(
        n_features=dim,
        lower_bounds=-5.0,
        upper_bounds=5.0,
    )
    
    # Fitness: Sphere function
    fitness_config = SphereConfig(num_objectives=1)
    fitness_evaluator = SphereEvaluator(fitness_config)
    
    # Selection operator
    if selection_type == "tournament":
        selector = TournamentSelection(tournament_size=selection_param)
    elif selection_type == "pool":
        selector = ElitePoolSelection(pool_size=selection_param)
    else:
        raise ValueError(f"Unknown selection_type: {selection_type}")
    
    # Crossover and mutation
    crossover = BlendCrossover(alpha=0.5, num_offspring=2)
    mutation = GaussianMutation(mutation_rate=mutation_rate, num_offspring=1, std_dev=0.1)
    
    # Engine
    engine = GeneticEngine(
        genome_config=genome_config,
        fitness_evaluator=fitness_evaluator,
        selector=selector,
        crossover=crossover,
        mutation=mutation,
        pop_size=pop_size,
        num_generations=num_generations,
        elite_size=1,
    )
    
    return engine

print("✓ Engine factory ready")

## 4. Experiment Runner

In [ ]:
def run_condition_experiment(
    selection_type: str,
    selection_param: int,
    pop_size: int,
    mutation_rate: float,
    n_runs: int = N_RUNS,
    num_generations: int = NUM_GENERATIONS,
    base_seed: int = GLOBAL_SEED,
) -> pd.DataFrame:
    """Run an experiment condition (one combo) using BenchmarkRunner.
    
    Returns:
        DataFrame with columns: selection_type, selection_param, pop_size, mutation_rate,
                                 run_id, seed, final_best, avg_per_gen_best
    """
    records = []
    seed_key = jr.PRNGKey(base_seed)
    
    # Generate seeds for all runs
    seeds = jr.split(seed_key, n_runs)
    
    for run_id, seed in enumerate(tqdm(seeds, desc=f"{selection_type}(param={selection_param})", leave=False)):
        seed_val = int(jr.randint(seed, (), 0, 2**31).item())
        
        # Create engine for this condition
        engine = make_genetic_engine(
            selection_type=selection_type,
            selection_param=selection_param,
            pop_size=pop_size,
            mutation_rate=mutation_rate,
            num_generations=num_generations,
        )
        
        # Run with BenchmarkRunner
        runner = BenchmarkRunner(
            engine=engine,
            experiment_name=f"sphere_{selection_type}_{selection_param}_pop{pop_size}_mu{mutation_rate}",
            output_dir=None,  # Skip artifact writing for speed
            write_artifacts=False,
        )
        
        result = runner.run(seeds=[seed_val])
        
        # Extract metrics from run
        run_data = result.runs[0]
        final_best = run_data.metrics.get("final_best", float('nan'))
        avg_per_gen = run_data.metrics.get("avg_best_per_gen", float('nan'))
        
        records.append({
            "selection_type": selection_type,
            "selection_param": selection_param,
            "pop_size": pop_size,
            "mutation_rate": mutation_rate,
            "run_id": run_id,
            "seed": seed_val,
            "final_best": final_best,
            "avg_per_gen_best": avg_per_gen,
        })
    
    return pd.DataFrame(records)

print("✓ Experiment runner ready")

## 5. Smoke Test (Quick Validation)

In [ ]:
# Quick test: 1 run each of tournament and pool
print("Running smoke test...")
df_tournament_smoke = run_condition_experiment(
    selection_type="tournament",
    selection_param=2,
    pop_size=50,
    mutation_rate=0.01,
    n_runs=1,
    num_generations=10,  # Short for smoke test
    base_seed=100,
)

df_pool_smoke = run_condition_experiment(
    selection_type="pool",
    selection_param=10,
    pop_size=50,
    mutation_rate=0.01,
    n_runs=1,
    num_generations=10,
    base_seed=101,
)

df_smoke = pd.concat([df_tournament_smoke, df_pool_smoke], ignore_index=True)
print("\n✓ Smoke test complete:")
print(df_smoke)

## 6. Full Experiment (Parameter Grid)

In [ ]:
run_full_experiment = False  # Set to True to run full grid

if run_full_experiment:
    print(f"Running full experiment grid...\n")
    t0 = time.time()
    
    all_records = []
    
    # Grid combinations
    combos = []
    for pop_size in POP_SIZES:
        for mutation_rate in MUTATION_RATES:
            for tournament_size in TOURNAMENT_SIZES:
                combos.append(("tournament", tournament_size, pop_size, mutation_rate))
            for pool_size in POOL_SIZES:
                combos.append(("pool", pool_size, pop_size, mutation_rate))
    
    total_combos = len(combos)
    print(f"Total combinations: {total_combos}")
    print(f"Total runs: {total_combos * N_RUNS}\n")
    
    for i, (sel_type, sel_param, pop, mu) in enumerate(tqdm(combos, desc="Overall progress")):
        df_condition = run_condition_experiment(
            selection_type=sel_type,
            selection_param=sel_param,
            pop_size=pop,
            mutation_rate=mu,
            n_runs=N_RUNS,
            num_generations=NUM_GENERATIONS,
            base_seed=GLOBAL_SEED + i * 1000,
        )
        all_records.append(df_condition)
    
    df_results = pd.concat(all_records, ignore_index=True)
    
    t1 = time.time()
    elapsed = (t1 - t0) / 60
    print(f"\n✓ Done in {elapsed:.2f} minutes")
    
    # Save results
    df_results.to_csv(RESULTS_CSV, index=False)
    print(f"✓ Results saved to {RESULTS_CSV}")
    print(f"\nDataFrame shape: {df_results.shape}")
    print(f"Columns: {list(df_results.columns)}")
else:
    print("Full experiment skipped. Set run_full_experiment=True to execute.")
    print(f"\nWhen enabled, will run: {len(list(range(len(POP_SIZES) * len(MUTATION_RATES) * (len(TOURNAMENT_SIZES) + len(POOL_SIZES)))))} conditions × {N_RUNS} runs")

## 7. Boxplot Visualization

In [ ]:
# Plot smoke test results
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 6))
ax = sns.boxplot(data=df_smoke, x="selection_type", y="final_best", palette="Set2")
ax.set_title("Smoke Test: Tournament vs Pool (1 run each, 10 gens)")
ax.set_ylabel("Final Best Fitness (lower is better)")
ax.set_xlabel("Selection Strategy")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "boxplot_smoke.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✓ Smoke test boxplot saved")

## 8. Load and Visualize Full Results (if available)

In [ ]:
if RESULTS_CSV.exists():
    df_all = pd.read_csv(RESULTS_CSV)
    print(f"✓ Loaded {len(df_all)} rows from {RESULTS_CSV}")
    
    # Overall boxplot
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.boxplot(data=df_all, x="selection_type", y="final_best", palette="Set1", ax=ax)
    ax.set_title(f"Engine Comparison: Tournament vs Pool ({N_RUNS} runs per config)")
    ax.set_ylabel("Final Best Fitness (lower is better)")
    ax.set_xlabel("Selection Strategy")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "boxplot_overall.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    # Summary statistics
    print("\n=== Summary Statistics ===")
    summary = df_all.groupby("selection_type")["final_best"].agg([
        ("count", "count"),
        ("mean", "mean"),
        ("median", "median"),
        ("std", "std"),
        ("min", "min"),
        ("max", "max"),
    ])
    print(summary)
else:
    print(f"Results CSV not found: {RESULTS_CSV}")
    print("Run the full experiment (set run_full_experiment=True) to generate results.")

## 9. Faceted Boxplots (by Parameter Grid)

In [ ]:
if RESULTS_CSV.exists():
    df_all = pd.read_csv(RESULTS_CSV)
    
    # Faceted plot: pop_size × mutation_rate
    g = sns.catplot(
        data=df_all,
        x="selection_type",
        y="final_best",
        col="pop_size",
        row="mutation_rate",
        kind="box",
        height=4,
        aspect=1.2,
        palette="Set2",
    )
    g.fig.suptitle("Engine Comparison Across Parameter Grid", y=1.00)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "boxplot_faceted.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ Faceted boxplot saved")
else:
    print("Full results not available yet.")

## 10. Statistical Tests (Mann–Whitney U)

In [ ]:
if RESULTS_CSV.exists():
    df_all = pd.read_csv(RESULTS_CSV)
    
    print("=== Mann–Whitney U Tests (Tournament vs Pool) ===")
    print("H0: Both strategies have same median final fitness\n")
    
    test_results = []
    
    for pop in sorted(df_all.pop_size.unique()):
        for mu in sorted(df_all.mutation_rate.unique()):
            subset = df_all[(df_all.pop_size == pop) & (df_all.mutation_rate == mu)]
            
            tournament_vals = subset[subset.selection_type == "tournament"]["final_best"].values
            pool_vals = subset[subset.selection_type == "pool"]["final_best"].values
            
            if len(tournament_vals) >= 2 and len(pool_vals) >= 2:
                stat, p_value = mannwhitneyu(tournament_vals, pool_vals, alternative="two-sided")
                significant = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
                
                tournament_median = np.median(tournament_vals)
                pool_median = np.median(pool_vals)
                
                test_results.append({
                    "pop_size": pop,
                    "mutation_rate": mu,
                    "tournament_median": tournament_median,
                    "pool_median": pool_median,
                    "p_value": p_value,
                    "significance": significant,
                })
                
                print(f"pop={pop:3d}, mu={mu:.2f}: p={p_value:.3e} {significant}  |  Tour={tournament_median:.4f}, Pool={pool_median:.4f}")
    
    # Summary table
    if test_results:
        df_tests = pd.DataFrame(test_results)
        df_tests.to_csv(RESULTS_DIR / "statistical_tests.csv", index=False)
        print(f"\n✓ Test results saved to {RESULTS_DIR / 'statistical_tests.csv'}")
else:
    print("Full results not available yet.")

## Summary

This notebook:
1. **Setup**: Configures two GA engines with TournamentSelection vs PoolSelection
2. **Smoke test**: Validates the setup with quick runs (1 run each)
3. **Full experiment** (optional): Runs parameter grid (pop_size × mutation_rate × selection_param) × 30 runs
4. **Visualization**: Generates boxplots comparing final fitness distributions
5. **Statistical analysis**: Mann–Whitney U tests for each (pop_size, mutation_rate) pair

**To run the full experiment**: Set `run_full_experiment = True` and re-run cell 6. Expect significant runtime.

**Results saved to**: `results/engine_comparison/`

# Title
# Engine comparison: Tournament vs Pool (Sphere)

# Notebook to compare GeneticEngine(TournamentSelection) vs GeneticEngine(ElitePoolSelection)
# on the Sphere problem. Defaults: 100 generations, grid of pop_size and mutation_rate,
# each experiment averages n_runs=30 runs per condition.


In [4]:
## 1) Imports and utilities

import time
from typing import Any, Dict, List, Tuple

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

# malthusjax imports
from malthusjax.engine.genetic_fastengine import GeneticEngine, GeneticEngineParams
from malthusjax.operators.selection.tournament import TournamentSelection
from malthusjax.operators.selection.elite_pool import ElitePoolSelection
from malthusjax.operators.crossover.real import BlendCrossover
from malthusjax.operators.mutation.real import GaussianMutation
from malthusjax.core.genome.real_genome import RealGenomeConfig
from malthusjax.core.fitness.real_evaluators import SphereConfig, SphereEvaluator

# For statistical test
from scipy.stats import mannwhitneyu


In [5]:
## 2) Config & seed

# Experiment defaults
NUM_GENERATIONS = 100  # confirmed
N_RUNS = 30  # runs per condition to form boxplots
POP_SIZES = [50, 100]
MUTATION_RATES = [0.01, 0.05]
TOURNAMENT_SIZES = [2, 4, 8]
POOL_SIZES = [10, 20, 50]
DIM = 10  # Sphere dimension

RESULTS_CSV = "results/engine_comparison_sphere.csv"

GLOBAL_SEED = 42
master_key = jr.PRNGKey(GLOBAL_SEED)


In [6]:
from joblib import Parallel, delayed


def run_experiments(
    grid: Dict[str, List[Any]],
    n_runs: int = N_RUNS,
    num_generations: int = NUM_GENERATIONS,
    base_seed: int = GLOBAL_SEED,
    parallel: bool = False,
    n_jobs: int = 4,
) -> pd.DataFrame:
    """Runs experiments for all grid combinations and returns a DataFrame of results.

    NOTE: The original implementation contained a broken `nonlocal global_idx` usage
    which caused a SyntaxError. This corrected version delegates to the safe
    serial runner `run_experiments_safe` when `parallel=False`.

    Parallel execution with JAX is non-trivial and is intentionally not
    implemented here; set `parallel=False` (default) to run safely.
    """
    if parallel:
        raise NotImplementedError(
            "Parallel execution is not supported in this notebook. Use run_experiments_safe instead."
        )

    # Normalize and delegate to the stable serial implementation
    grid_local = {
        "pop_size": grid["pop_size"],
        "mutation_rate": grid["mutation_rate"],
        "tournament_size": grid["tournament_size"],
        "pool_size": grid["pool_size"],
        "dim": grid.get("dim", DIM),
    }

    return run_experiments_safe(grid_local, n_runs=n_runs, num_generations=num_generations, base_seed=base_seed)


def _serial_run(sel_type, sel_param, pop, mu, run_idx, seeds, seed_idx, num_generations):
    # helper for Parallel use - NOT fully used in this notebook to avoid JAX parallel caveats
    seed_val = int(jr.randint(seeds[run_idx], (), 0, 2**31).item())
    eng = make_engine(sel_type, sel_param, pop, mu, num_generations)
    return run_single_run(seed_val, eng)


In [7]:
## 3b) Safe serial experiment runner (recommended)

def run_experiments_safe(grid: Dict[str, List[Any]], n_runs: int = N_RUNS, num_generations: int = NUM_GENERATIONS, base_seed: int = GLOBAL_SEED) -> pd.DataFrame:
    """Serial runner that is robust to JAX compilation issues.

    Returns a DataFrame with one row per run.
    """
    records = []
    seed_key = jr.PRNGKey(base_seed)
    combos = []
    for pop in grid["pop_size"]:
        for mu in grid["mutation_rate"]:
            for t in grid["tournament_size"]:
                combos.append(("tournament", t, pop, mu))
            for p in grid["pool_size"]:
                combos.append(("pool", p, pop, mu))

    total_runs = len(combos) * n_runs
    print(f"Running {len(combos)} combos x {n_runs} runs = {total_runs} runs (serial)")

    # generate seeds
    seeds = jr.split(seed_key, total_runs + 1)
    seed_ptr = 0

    for sel_type, sel_param, pop, mu in tqdm(combos, desc="Combos"):
        for i in range(n_runs):
            seed_val = int(jr.randint(seeds[seed_ptr], (), 0, 2**31).item())
            seed_ptr += 1
            eng = make_engine(sel_type, sel_param, pop, mu, num_generations)
            v = run_single_run(seed_val, eng)
            records.append({
                "engine": "GeneticEngine",
                "selection_type": sel_type,
                "selection_param": sel_param,
                "pop_size": pop,
                "mutation_rate": mu,
                "run_id": i,
                "final_best": v,
                "pop_dim": grid.get("dim", DIM),
                "num_generations": num_generations,
            })

    df = pd.DataFrame.from_records(records)
    return df


In [8]:
## 4) Smoke test (quick) — run a tiny experiment to validate the setup

small_grid = {
    "pop_size": [50],
    "mutation_rate": [0.01],
    "tournament_size": [2],
    "pool_size": [10],
    "dim": DIM,
}

print("Running smoke test: 3 runs x 1 combo x 10 gens")
df_smoke = run_experiments_safe(small_grid, n_runs=3, num_generations=10, base_seed=100)
print(df_smoke)

# Basic aggregate
print(df_smoke.groupby(["selection_type"]).final_best.describe())


Running smoke test: 3 runs x 1 combo x 10 gens
Running 2 combos x 3 runs = 6 runs (serial)


Combos:   0%|          | 0/2 [00:00<?, ?it/s]


OverflowError: An overflow was encountered while parsing an argument to a jitted computation, whose argument path is maxval.

In [ ]:
## 5) Plotting helper: boxplots per parameter combo

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

def plot_boxplot_for_df(df: pd.DataFrame, title: str = "Engine comparison"):
    plt.figure(figsize=(6, 5))
    ax = sns.boxplot(data=df, x="selection_type", y="final_best")
    ax.set_title(title)
    ax.set_ylabel('Final Sphere value (lower is better)')
    return ax

# Quick plot for smoke
plot_boxplot_for_df(df_smoke, title="Smoke test: Tournament vs Pool (3 runs)")
plt.show()

# If a full results CSV exists, load and show an example faceted plot
import os
if os.path.exists(RESULTS_CSV):
    df_all = pd.read_csv(RESULTS_CSV)
    # Example: facet by pop_size and mutation_rate, showing selection_type distributions
    g = sns.catplot(data=df_all, x='selection_type', y='final_best', col='pop_size', row='mutation_rate', kind='box', height=3.5)
    g.fig.suptitle('Engine comparison across parameter grid', y=1.02)
    plt.show()


In [ ]:
## 6) Full run (disabled by default) and saving

run_full = False  # Set to True to execute the full grid (may take long)
if run_full:
    grid = {
        "pop_size": POP_SIZES,
        "mutation_rate": MUTATION_RATES,
        "tournament_size": TOURNAMENT_SIZES,
        "pool_size": POOL_SIZES,
        "dim": DIM,
    }
    t0 = time.time()
    df_results = run_experiments_safe(grid, n_runs=N_RUNS, num_generations=NUM_GENERATIONS, base_seed=GLOBAL_SEED)
    t1 = time.time()
    print(f"Done: {(t1 - t0)/60:.2f} min")

    os.makedirs('results', exist_ok=True)
    df_results.to_csv(RESULTS_CSV, index=False)
    print(f"Saved results to {RESULTS_CSV}")
else:
    print("Full run skipped. Set run_full=True to run full experiments.")


In [ ]:
## 7) Statistical tests: Mann–Whitney U per (pop, mutation_rate)

if os.path.exists(RESULTS_CSV):
    df_all = pd.read_csv(RESULTS_CSV)
    for pop in sorted(df_all.pop_size.unique()):
        for mu in sorted(df_all.mutation_rate.unique()):
            t_vals = df_all[(df_all.pop_size == pop) & (df_all.mutation_rate == mu) & (df_all.selection_type == 'tournament')]['final_best']
            p_vals = df_all[(df_all.pop_size == pop) & (df_all.mutation_rate == mu) & (df_all.selection_type == 'pool')]['final_best']
            if len(t_vals) >= 2 and len(p_vals) >= 2:
                stat, p = mannwhitneyu(t_vals, p_vals, alternative='two-sided')
                print(f"pop={pop}, mu={mu}: tournament n={len(t_vals)}, pool n={len(p_vals)}, p={p:.3e}")
else:
    print("No results CSV found - run full experiment and save results to enable statistical tests.")


In [ ]:
## 8) Unit tests (quick smoke)

def test_make_engine():
    e = make_engine('tournament', 2, 50, 0.01, 5)
    assert isinstance(e, GeneticEngine)


def test_run_single_run():
    e = make_engine('pool', 10, 50, 0.01, 5)
    v = run_single_run(999, e)
    assert np.isfinite(v) and v >= 0.0

print("Two quick tests defined. Run `pytest -q <this-notebook>.py` or convert into test file in tests/ to execute.")


## Conclusions & Next steps

- This notebook compares GeneticEngine with TournamentSelection vs ElitePoolSelection on the Sphere function.
- Run the smoke test first (cell above). To execute the full grid, set `run_full = True` in cell "Full run" and re-run that cell. Expect the full run to take significant time (compilation + runs).
- Tips:
  - Consider reducing `N_RUNS` while prototyping.
  - Use `run_experiments_safe` to avoid JAX parallelization pitfalls; parallel execution is experimental.
  - Results CSV saved to `results/engine_comparison_sphere.csv` for re-use.

Happy benchmarking! ✅
